# Bài tập — Thị giác máy tính (121036)

## CLAHE · Ngưỡng thích nghi bằng ảnh tích phân · Hough đa tỉ lệ

**Nhóm:**  
**Thành viên (MSSV — Họ tên):**
1.
2.
3.

---

### Quy định về việc dùng công cụ AI

| | **Phần cài đặt** | **Phần thí nghiệm minh họa** |
|---|---|---|
| **Nội dung** | Các hàm trong mục `Cài đặt` của mỗi kỹ thuật | Chọn ảnh, dựng bộ ảnh, quét tham số, vẽ hình, so sánh với thuật toán đã học |
| **Công cụ AI** | ❌ **Không được dùng** — kể cả để gợi ý, sửa lỗi hay giải thích mã | ✅ **Được dùng** — để sinh/chọn ảnh phù hợp và dựng khung thí nghiệm |
| **Ràng buộc** | Tự viết hoàn toàn, chỉ dùng thư viện trong danh sách cho phép | **Mọi prompt đã dùng phải được chép nguyên văn vào notebook**, ngay trên ô mã tương ứng |

Trước **mỗi** ô mã ở phần thí nghiệm có dùng AI hỗ trợ, chèn một ô markdown theo đúng mẫu ở ô kế tiếp.
Ô mã nào không có ô markdown này được hiểu là do các em tự viết.
**Thí nghiệm dùng AI mà không khai báo prompt sẽ không được tính điểm cho phần đó.**

### ⚙️ Có dùng AI hỗ trợ

**Công cụ:** *(tên công cụ, phiên bản nếu biết)*

**Mục đích:** *(sinh ảnh thử / dựng vòng lặp quét tham số / vẽ lưới ảnh / ...)*

**Prompt:**

> *(chép nguyên văn prompt đã dùng — không tóm tắt, không viết lại)*

**Đã sửa lại những gì:** *(mô tả ngắn phần các em phải sửa để chạy đúng)*

*(Đây là ô mẫu — sao chép ô này và điền vào mỗi chỗ cần khai báo. Giữ nguyên ô mẫu này ở đây.)*

---
## 0. Chuẩn bị

Thư viện được phép dùng trong **phần cài đặt**: `numpy`, `matplotlib`, `time`/`timeit`.

`cv2`, `scipy`, `skimage` chỉ được gọi trong các ô có nhãn **`ĐỐI CHIẾU`** hoặc trong phần
thí nghiệm khi so sánh với thuật toán đã học. Không được gọi chúng bên trong các hàm phải nộp.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["image.cmap"]     = "gray"
plt.rcParams["image.interpolation"] = "nearest"

### 0.1 Tiện ích hiển thị *(cung cấp sẵn — không cần sửa)*

`show_grid` dùng để trình bày lưới ảnh khi quét tham số.

In [ ]:
def show_grid(images, titles=None, ncols=4, figw=3.0, suptitle=None, vmin=0, vmax=255):
    """Hiển thị một danh sách ảnh xám thành lưới.

    images : list các mảng 2 chiều
    titles : list nhãn tương ứng (có thể None)
    """
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figw * ncols, figw * nrows * 1.12))
    axes = np.atleast_1d(axes).ravel()
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < n:
            ax.imshow(images[i], vmin=vmin, vmax=vmax)
            if titles is not None:
                ax.set_title(titles[i], fontsize=9)
    if suptitle:
        fig.suptitle(suptitle, fontsize=12)
    fig.tight_layout()
    plt.show()


def show_hist(img, ax=None, title=None):
    """Vẽ lược đồ xám của một ảnh uint8."""
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 2.2))
    ax.hist(np.asarray(img).ravel(), bins=256, range=(0, 255), color="#195AA0")
    ax.set_xlim(0, 255)
    ax.set_yticks([])
    if title:
        ax.set_title(title, fontsize=9)
    return ax

### 0.2 Ảnh dùng trong bài

Nạp ảnh của nhóm vào đây. Yêu cầu về bộ ảnh được nêu riêng ở phần thí nghiệm của từng kỹ thuật —
**đọc trước** rồi hãy chọn ảnh, vì mỗi kỹ thuật cần một loại ảnh khác nhau để hiệu ứng lộ ra được.

Nếu nạp ảnh màu, chuyển sang ảnh xám `uint8` trước khi dùng.

In [ ]:
# TODO: nạp ảnh của nhóm.
#   img = ...   # mảng (H, W) dtype uint8

IMAGES = {}   # ví dụ: {"nhan_container": img1, "chung_tu": img2}

---
---
# 1. Kỹ thuật 1 — CLAHE

Lý thuyết: xem mục 2 của tài liệu bổ trợ.

## 1.1 Cài đặt — ❌ KHÔNG dùng AI

Thư viện được phép: `numpy` (kể cả `np.cumsum`, `np.bincount`, `np.histogram`), `matplotlib`.

Bị cấm: `cv2.createCLAHE`, `cv2.equalizeHist`, `skimage.exposure.equalize_adapthist`,
`scipy.ndimage.map_coordinates`.

In [ ]:
def clip_histogram(hist, clip_limit):
    """Cắt và phân phối lại lược đồ xám.

    hist       : mảng (L,) số nguyên — lược đồ của MỘT ô.
    clip_limit : ngưỡng cắt. Ghi rõ ở đây quy ước các em dùng
                 (beta tuyệt đối, hay hệ số s_max như OpenCV).

    Trả về     : mảng (L,) đã cắt, tổng giữ nguyên (sai số <= L do làm tròn).
    """
    raise NotImplementedError

In [ ]:
def build_tile_luts(img, grid=(8, 8), clip_limit=2.0):
    """Dựng bảng tra cho từng ô của lưới.

    img   : (H, W) uint8
    grid  : (Gy, Gx) số ô theo hàng và cột

    Trả về: mảng (Gy, Gx, L) uint8 — bảng tra của từng ô.
    """
    raise NotImplementedError

In [ ]:
def clahe(img, grid=(8, 8), clip_limit=2.0):
    """CLAHE đầy đủ: dựng bảng tra từng ô, rồi nội suy song tuyến GIỮA CÁC BẢNG TRA.

    Lưu ý: nội suy giá trị bảng tra T_ij[v] (bốn số vô hướng), KHÔNG nội suy giữa bốn ảnh.

    img   : (H, W) uint8
    Trả về: (H, W) uint8
    """
    raise NotImplementedError

**Ô kiểm tra nhanh trong lúc viết** *(tùy chọn, không chấm điểm — dùng để tự dò lỗi)*

In [ ]:
# Gợi ý dùng khi đang gỡ lỗi:
#   - vẽ một lát cắt ngang của ảnh ra: nếu thấy bậc nhảy tuần hoàn đúng bằng bề rộng ô
#     thì phần nội suy đang sai.
#   - kiểm tra mọi bảng tra không giảm.

## 1.2 Thí nghiệm minh họa — ✅ được dùng AI *(nhớ khai báo prompt)*

### (a) Ảnh hưởng của tham số

Quét `clip_limit` và `grid` qua **ít nhất bốn giá trị mỗi tham số**, trình bày dạng lưới ảnh.

Chọn ảnh sao cho hiệu ứng **nhìn thấy được**:
- `clip_limit` chỉ lộ tác dụng trên ảnh có **vùng phẳng nhiều nhiễu**;
- `grid` chỉ lộ tác dụng trên ảnh có **ánh sáng không đều theo vùng**.

Có thể phải dùng hai ảnh khác nhau cho hai tham số.

In [ ]:
# TODO (a): quét clip_limit

In [ ]:
# TODO (a): quét grid

**Nhận xét (a):**

*(Viết ngắn: mỗi tham số điều khiển hiện tượng gì, và vì sao phải chọn ảnh như trên
thì hiện tượng đó mới lộ ra.)*

### (b) So sánh với thuật toán đã học

Đặt cạnh nhau, **trên cùng một ảnh**:

1. kéo giãn tương phản tuyến tính
2. hiệu chỉnh gamma
3. cân bằng lược đồ xám toàn cục (HE)
4. CLAHE

Kèm **lược đồ xám của từng kết quả** (dùng `show_hist`).

Bắt buộc chỉ ra:
- một ảnh mà **CLAHE thắng rõ rệt**;
- một ảnh mà **CLAHE thua** một trong ba cách kia — và giải thích vì sao.

In [ ]:
# TODO (b): bốn phương pháp cạnh nhau + lược đồ xám
# (HE toàn cục có thể gọi cv2.equalizeHist ở đây — đây là ô ĐỐI CHIẾU, không phải ô cài đặt)

In [ ]:
# TODO (b): ảnh mà CLAHE THUA

**Nhận xét (b):**

*(CLAHE thua ở loại ảnh nào? Cơ chế nào của CLAHE gây ra điều đó?)*

---
---
# 2. Kỹ thuật 2 — Ngưỡng thích nghi bằng ảnh tích phân

Lý thuyết: xem mục 3 của tài liệu bổ trợ.

## 2.1 Cài đặt — ❌ KHÔNG dùng AI

Thư viện được phép: `numpy` (kể cả `np.cumsum`), `time`/`timeit`, `matplotlib`.

Bị cấm: `cv2.integral`, `cv2.adaptiveThreshold`, `cv2.boxFilter`, `cv2.blur`,
`scipy.ndimage.uniform_filter`, `skimage.filters.threshold_sauvola`.

In [ ]:
def integral_image(img):
    """Ảnh tích phân có đệm viền.

    img   : (H, W)
    Trả về: (H+1, W+1) dtype int64, hàng 0 và cột 0 bằng 0.
    """
    raise NotImplementedError

In [ ]:
def box_sum(S, y1, x1, y2, x2):
    """Tổng vùng [y1..y2, x1..x2] (bao gồm cả biên) từ ảnh tích phân đã đệm viền.

    Phải vectơ hóa được: y1, x1, y2, x2 có thể là mảng cùng shape.
    """
    raise NotImplementedError

In [ ]:
def local_mean_std(img, radius):
    """Trung bình và độ lệch chuẩn cục bộ trong cửa sổ (2r+1) x (2r+1).

    Dùng HAI ảnh tích phân: của img và của img^2.
    Nhớ kẹp phương sai về không âm trước khi lấy căn.

    Trả về: (mean, std), mỗi cái (H, W) float64.
    """
    raise NotImplementedError

In [ ]:
def adaptive_threshold(img, radius=15, method="bradley", t=0.15, k=0.34, R=128.0):
    """Nhị phân hóa bằng ngưỡng thích nghi.

    method : "bradley"  ->  T = m * (1 - t)
             "sauvola"  ->  T = m * (1 + k * (s/R - 1))

    Trả về : (H, W) uint8, giá trị 0 hoặc 255.
    """
    raise NotImplementedError

## 2.2 Thí nghiệm minh họa — ✅ được dùng AI *(nhớ khai báo prompt)*

### (a) Ảnh hưởng của tham số

Quét `radius` qua **ít nhất năm giá trị** và `t` (hoặc `k`) qua **ít nhất ba giá trị**.

Hai hiện tượng phải xuất hiện được trong lưới ảnh:
- `radius` **quá nhỏ** → nét chữ dày bị **rỗng ruột**;
- `radius` **quá lớn** → phương pháp thoái hóa dần về **ngưỡng toàn cục**.

Chỉ ra vùng giá trị hợp lý và giải thích nó phụ thuộc vào **cái gì trong ảnh**.

In [ ]:
# TODO (a): quét radius

In [ ]:
# TODO (a): quét t (hoặc k)

**Nhận xét (a):**

*(Vùng `radius` hợp lý phụ thuộc vào đại lượng nào của ảnh? Nếu chụp cùng cảnh ở độ phân giải
gấp đôi thì `radius` tốt thay đổi thế nào?)*

### (b) So sánh với thuật toán đã học

Trên **cùng một ảnh**: ngưỡng cố định thủ công, **Otsu toàn cục**, **Bradley–Roth**, **Sauvola**.

⚠️ Ảnh dùng để so sánh **phải có ánh sáng không đều**. Trên ảnh chiếu sáng đều, cả bốn cho kết quả
gần như nhau và phép so sánh trở nên vô nghĩa.

Bổ sung **một ảnh có vùng nền phẳng rộng** để cho thấy chỗ Sauvola hơn hẳn Bradley–Roth.

In [ ]:
# TODO (b): bốn phương pháp cạnh nhau (Otsu có thể gọi cv2 — ô ĐỐI CHIẾU)

In [ ]:
# TODO (b): ảnh có nền phẳng rộng — Sauvola vs Bradley-Roth

**Nhận xét (b):**

### (c) Đường cong thời gian

Ba cách tính trung bình cục bộ, với `radius` $\in \{1, 2, 4, 8, 16, 32, 64\}$:

| Cách làm | Chi phí / điểm ảnh |
|---|---|
| Tích chập trực tiếp với nhân hộp *(dùng `convolve2d` của chính nhóm)* | $O(r^2)$ |
| Tích chập tách được (hai nhân 1 chiều) | $O(r)$ |
| Ảnh tích phân | $O(1)$ |

Vẽ ba đường trên cùng một trục, trục $y$ thang log.
Đường thứ ba phải **nằm ngang** — nếu không, gần như chắc chắn còn sót một vòng lặp Python
theo điểm ảnh ở đâu đó.

In [ ]:
# TODO (c): đo thời gian và vẽ ba đường (thang log)
RADII = [1, 2, 4, 8, 16, 32, 64]

**Nhận xét (c):**

---
---
# 3. Kỹ thuật 3 — Biến đổi Hough đa tỉ lệ

Lý thuyết: xem mục 4 của tài liệu bổ trợ.

Kỹ thuật này cần hàm `convolve2d` do chính nhóm viết (dùng để làm mượt mảng tích lũy).
Dán hàm đó vào ô dưới đây.

In [ ]:
# TODO: dán hàm convolve2d của nhóm vào đây

## 3.1 Cài đặt — ❌ KHÔNG dùng AI

Thư viện được phép: `numpy`, `convolve2d` của chính nhóm, `matplotlib`.

Bị cấm: `cv2.HoughLines`, `cv2.HoughLinesP`, `skimage.transform.hough_line`,
`skimage.transform.hough_line_peaks`, `scipy.signal.find_peaks` (áp cho mảng 2 chiều),
`scipy.ndimage.maximum_filter`.

In [ ]:
def hough_accumulate(edges, theta_range, rho_range, n_theta, n_rho,
                     grad_dir=None, delta_deg=None):
    """Bỏ phiếu vào mảng tích lũy Hough.

    edges       : (H, W) bool hoặc uint8 — bản đồ biên.
    theta_range : (theta_min, theta_max) radian.
    rho_range   : (rho_min, rho_max).
    grad_dir    : (H, W) float — hướng gradient tại mỗi điểm, radian, đã chuẩn hóa về [0, pi).
                  Nếu khác None, mỗi điểm chỉ bỏ phiếu trong dải +/- delta_deg quanh hướng đó.

    Trả về      : (A, thetas, rhos)
    """
    raise NotImplementedError

In [ ]:
def find_peaks(A, n_peaks, nms_radius=3, smooth_sigma=1.0):
    """Tìm đỉnh trong mảng tích lũy.

    Ba bước: làm mượt A bằng convolve2d của chính nhóm -> triệt phi cực đại trong bán kính
    nms_radius -> lấy n_peaks đỉnh cao nhất.

    Trả về: list [(i_rho, i_theta, votes), ...] sắp giảm dần theo votes.
    """
    raise NotImplementedError

In [ ]:
def hough_multiscale(edges, grad_dir=None, n_peaks=8, n_levels=2, refine_factor=8,
                     n_theta0=180, n_rho0=None, delta_deg=None):
    """Hough thô-đến-mịn.

    Tầng 0 dựng mảng tích lũy thô. Với mỗi đỉnh ứng viên, dựng lại một mảng nhỏ và mịn
    chỉ phủ lân cận đỉnh đó, bước chia nhỏ đi refine_factor lần. Lặp n_levels tầng.

    CẢNH BÁO: khi mở cửa sổ lân cận quanh theta ~ 0 hoặc theta ~ pi, phần vượt biên phải
    quấn vòng KÈM THEO ĐỔI DẤU rho.

    Trả về: list [(rho, theta, votes), ...]
    """
    raise NotImplementedError

### Tiện ích: sinh ảnh tổng hợp có đáp án đúng *(cung cấp sẵn)*

Kỹ thuật này là kỹ thuật **duy nhất** trong ba kỹ thuật cho phép tạo ảnh có đáp án đúng tuyệt đối.
Hãy tận dụng: phần lớn thí nghiệm nên chạy trên ảnh tổng hợp trước, rồi mới kiểm lại trên ảnh thật.

In [ ]:
def draw_line(canvas, rho, theta_deg, length=None, value=255, thickness=1):
    """Vẽ một đoạn thẳng có (rho, theta) đã biết lên canvas, tính từ gốc (0, 0) ở góc trên-trái.

    rho, theta_deg : tham số đường thẳng, theo đúng quy ước rho = x cos(theta) + y sin(theta).
    length         : độ dài đoạn (điểm ảnh). None = kéo hết ảnh.

    Trả về canvas (sửa tại chỗ).
    """
    H, W = canvas.shape
    th = np.deg2rad(theta_deg)
    ct, st = np.cos(th), np.sin(th)
    x0, y0 = rho * ct, rho * st          # chân đường vuông góc từ gốc
    dx, dy = -st, ct                     # vectơ chỉ phương
    L = length if length is not None else 2 * int(np.hypot(H, W))
    t = np.arange(-L / 2, L / 2, 0.5)
    xs = np.round(x0 + t * dx).astype(int)
    ys = np.round(y0 + t * dy).astype(int)
    for oy in range(-(thickness // 2), thickness // 2 + 1):
        for ox in range(-(thickness // 2), thickness // 2 + 1):
            X, Y = xs + ox, ys + oy
            m = (X >= 0) & (X < W) & (Y >= 0) & (Y < H)
            canvas[Y[m], X[m]] = value
    return canvas


def synthetic_lines(shape, lines, noise_frac=0.0, seed=0):
    """Ảnh nhị phân chứa các đoạn thẳng đã biết.

    lines : list các dict {"rho": .., "theta_deg": .., "length": .. (tùy chọn)}
    noise_frac : tỉ lệ điểm biên nhiễu thêm vào, tính trên số điểm biên thật.

    Trả về (img, lines) — lines chính là đáp án đúng.
    """
    rng = np.random.default_rng(seed)
    img = np.zeros(shape, np.uint8)
    for ln in lines:
        draw_line(img, ln["rho"], ln["theta_deg"], ln.get("length"))
    if noise_frac > 0:
        n = int(noise_frac * np.count_nonzero(img))
        ys = rng.integers(0, shape[0], n)
        xs = rng.integers(0, shape[1], n)
        img[ys, xs] = 255
    return img, lines

In [ ]:
# Ví dụ dùng thử tiện ích (không phải bài làm):
demo, gt = synthetic_lines((300, 400), [
    {"rho": 137.0, "theta_deg": 41.3},
    {"rho":  80.0, "theta_deg":  0.4},   # gần thẳng đứng — bẫy quấn vòng
], noise_frac=0.15, seed=1)
plt.imshow(demo); plt.title("ảnh tổng hợp có đáp án đúng"); plt.axis("off"); plt.show()

## 3.2 Thí nghiệm minh họa — ✅ được dùng AI *(nhớ khai báo prompt)*

### (a) Ảnh hưởng của tham số

Quét $\Delta\theta$ và $\Delta\rho$ qua **ít nhất bốn giá trị**.
Trình bày **cả mảng tích lũy** (dưới dạng ảnh) **và** các đường thẳng phát hiện được —
mảng tích lũy là nơi hiện tượng **vỡ phiếu** nhìn thấy rõ nhất.

Sau đó quét số tầng $n$ và hệ số tinh chỉnh $r$, vẽ **sai số góc theo $n$**.

In [ ]:
# TODO (a): quét độ phân giải, hiển thị mảng tích lũy + đường phát hiện

In [ ]:
# TODO (a): sai số góc theo số tầng n

**Nhận xét (a):**

*(Sai số góc giảm theo hệ số nào mỗi tầng? Nó chạm sàn ở đâu, và cái sàn đó do cái gì quyết định?)*

### (b) So sánh với thuật toán đã học

Trên **cùng bộ ảnh**:

1. Hough chuẩn tắc ở độ phân giải **thô**
2. Hough chuẩn tắc ở độ phân giải **mịn**
3. **Hough đa tỉ lệ**

So sánh theo **ba chỉ số**: sai số góc so với đáp án đúng, tỉ số đỉnh trên nền của mảng tích lũy,
và thời gian chạy. Trình bày dạng bảng.

Bổ sung một lần chạy **bật/tắt bỏ phiếu theo hướng gradient** trên cùng ảnh.

In [ ]:
# TODO (b): bảng so sánh ba chỉ số

In [ ]:
# TODO (b): bật/tắt bỏ phiếu theo hướng gradient

**Nhận xét (b):**

### (c) Hai trường hợp bắt buộc có trong bộ ảnh

1. Một ảnh chứa **đường gần thẳng đứng** ($\theta_0$ quanh $0^\circ$ hoặc $179^\circ$).
2. Một ảnh chứa **đồng thời đoạn rất dài và đoạn rất ngắn**.

Cả hai đều là chỗ Hough một tỉ lệ hỏng. Chỉ rõ nó hỏng như thế nào.

In [ ]:
# TODO (c): trường hợp đường gần thẳng đứng

In [ ]:
# TODO (c): trường hợp đoạn dài + đoạn ngắn

**Nhận xét (c):**

---
---
## Trước khi nộp

- [ ] Ba mục **Cài đặt** không còn `raise NotImplementedError`, và không có ô nào trong đó gọi
      `cv2` / `scipy` / `skimage`.
- [ ] Mỗi ô mã ở phần thí nghiệm có dùng AI đều có ô khai báo prompt ngay phía trên.
- [ ] Notebook chạy được từ đầu đến cuối (`Kernel → Restart & Run All`) không lỗi.
- [ ] Ảnh dùng trong bài đã được nộp kèm, hoặc mã sinh ảnh nằm ngay trong notebook.
- [ ] Đã điền tên nhóm và danh sách thành viên ở đầu notebook.